In [1]:
# kernel: thesis_clean4
#https://github.com/scaomath/fourier_neural_operator/blob/master/fourier_1d.py
import matplotlib.pyplot as plt
import pandas as pd
#from neuralop.models import FNO
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.functional as F
from torch.nn.parameter import Parameter

In [2]:
import torch
print(torch.__version__)

2.5.1+cu121


In [12]:
#data = pyscrew.get_data(scenario="s02", handle_duplicates="first", handle_missings="mean",force_download=True, target_length=800)
#df = pd.DataFrame(data)
#df.to_pickle("screw_data_s02-v2_identical-to-v1.pkl")



2026-06-27 16:10:02 - INFO - pyscrew.main - Starting data retrieval for scenario: s02 (surface-friction)
2026-06-27 16:10:02 - INFO - pyscrew.pipeline.loading - Using cache directory (absolute): c:\Users\Patrick\miniconda3\envs\thesis_clean4\Lib\site-packages\pyscrew\downloads
2026-06-27 16:10:02 - INFO - pyscrew.pipeline.loading - Beginning data extraction for scenario 's02_variations-in-surface-friction.zip' (force=True)
2026-06-27 16:10:02 - INFO - pyscrew.pipeline.loading - Downloading dataset 's02_variations-in-surface-friction.zip' from Zenodo URL: https://zenodo.org/records/16031381/files/s02_variations-in-surface-friction.zip?download=1


2026-06-27 16:10:28 - INFO - pyscrew.pipeline.loading - Download of 's02_variations-in-surface-friction.zip' completed (83,946,972 bytes). Beginning checksum verification...
2026-06-27 16:10:28 - INFO - pyscrew.pipeline.loading - Verifying MD5 checksum for 's02_variations-in-surface-friction.zip'...
2026-06-27 16:10:28 - INFO - pyscrew.pipeline.loading - Checksum verification successful for 's02_variations-in-surface-friction.zip' (MD5: 0bc948a6e8c6e83f72dbe36973131558)


2026-06-27 16:10:32 - INFO - pyscrew.pipeline.loading - Extracting archive 's02_variations-in-surface-friction.zip' to directory: c:\Users\Patrick\miniconda3\envs\thesis_clean4\Lib\site-packages\pyscrew\downloads\extracted\s02_variations-in-surface-friction
2026-06-27 16:10:38 - INFO - pyscrew.pipeline.loading - Extraction of 's02_variations-in-surface-friction.zip' completed successfully to: c:\Users\Patrick\miniconda3\envs\thesis_clean4\Lib\site-packages\pyscrew\downloads\extracted\s02_variations-in-surface-friction
2026-06-27 16:10:38 - INFO - pyscrew.core.dataset - Selected 12500 files
2026-06-27 16:10:49 - INFO - pyscrew.core.dataset - Successfully loaded 12500 screw runs
2026-06-27 16:10:49 - INFO - pyscrew.pipeline.processing - Adding input_logging transformer to pipeline
2026-06-27 16:10:49 - INFO - pyscrew.pipeline.processing - Adding step_unpacking transformer to pipeline
2026-06-27 16:10:49 - INFO - pyscrew.pipeline.processing - Adding duplicate handling with first
2026-06-2

2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Completed missing interpolation using 'mean' method (interval=0.0012)
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Processed 12,500 series with 8,619,982 total points
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Found gaps - min: 0.0012s, max: 0.1128s, avg: 0.0013s
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Added 484,205 points (+5.62% of total)
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Average 38.7 points added per series


2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - Starting to apply equal lengths.
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'target_length' : 800
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'padding_value' : 0.0
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'padding_position' : post
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'cutoff_position' : post
2026-06-27 16:12:48 - INFO - pyscrew.pipeline.transformers.handle_lengths - Finished applying equal lengths to the screw driving data.
2026-06-27 16:12:48 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Total screw runs loaded:	12500
2026-06-27 16:12:48 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Average change of length:	728.33 -> 800.00
2026-06-27 16:12:48 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Total points before normalization:	9,104,

In [3]:
df = pd.read_pickle("screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [9]:
class Anziehdrehmoment(nn.Module):
    def __init__(self):
        super().__init__()

        self.pitch = torch.tensor(0.00146, dtype=torch.float32)
        #äußere durchmesser des schraubteils
        self.D = torch.tensor(0.004, dtype=torch.float32)
        #d2 durchmesser
        self.d2 = self.D - 0.6495 * self.pitch
        #innere durchmesser schraube
        self.d3 = torch.tensor(0.00281, dtype=torch.float32)
        #self.screw_length = torch.tensor(0.012, dtype=torch.float32)
        # d_K = Kopfdurchmesser der Schraube 0.009 und d_R = Schraubendurchmesser 0.004 d_K_R = d_K + d_R / 2
        self.d_K_R = torch.tensor(0.009 + 0.004 / 2, dtype=torch.float32)

        #Gewindereibwinkel Parameter
        #self.mu_G = nn.Parameter(torch.tensor([0.12, 0.14, 0.16, 0.18,0.20, 0.22, 0.24, 0.26], dtype=torch.float32))
        self.mu_G = torch.tensor([0.1428, 0.1835, 0.2183, 0.2527, 0.2869, 0.3192, 0.3500, 0.3825], dtype=torch.float32)
        
        #Kopfreibwinkel parameter
        #self.mu_K = nn.Parameter(torch.tensor([0.12, 0.14, 0.16, 0.18,0.20, 0.22, 0.24, 0.26], dtype=torch.float32))
        self.mu_K = torch.tensor([-0.0369, -0.0423, -0.0467, -0.0526, -0.0563, -0.0609, -0.0648, -0.0696], dtype=torch.float32)
        # zulässige Spannung = 0.9 * Streckgrenze 
        # Re Streckgrenze von Stahlschraube m4 10.9 = 900 MPa 
        self.zul_spannung = torch.tensor(0.9 * 900e6, dtype=torch.float32)

        # A_S = pi/4 * (d2 + d3 / 2)**2
        self.A_S = (torch.pi / 4) * ((self.d2 + self.d3) / 2) ** 2

    def forward(self, logits, phase, angle, X_batch, class_input):

        device = X_batch.device
        probs = torch.softmax(logits, dim=-1)
        #parameter auf Device 
        #mu_K = self.mu_K[class_input].to(device)
        #mu_G = self.mu_G[class_input].to(device)
        
        #feste
        mu_K = self.mu_K.to(device)
        mu_G = self.mu_G.to(device)
        mu_K = torch.sum(probs * mu_K, dim=-1)
        mu_G = torch.sum(probs * mu_G, dim=-1)
        

        #reshape für seq + batch
        mu_K = mu_K[:, None, None]
        mu_G = mu_G[:, None, None]

        # konstanten auf device
        d2 = self.d2.to(device)
        d_K_R = self.d_K_R.to(device)
        pitch = self.pitch.to(device)
        A_S = self.A_S.to(device)

        # Gewindereibwinkel archtan(mu_G / cos(beta/2)) beta = 20°
        winkel = torch.deg2rad(torch.tensor(20.0, device=device))
        gewindereibwinkel = torch.atan(mu_G / torch.cos(winkel / 2))
        
        #Steigungswinkel (archtan(P/d2 * pi))
        steigungswinkel = torch.atan(pitch / (torch.pi * d2))

        #Spannungsdurchmesser ds = wurzel((A_S * 4/pi))
        d_s = torch.sqrt(A_S * 4 / torch.pi)
        W_p = (torch.pi * d_s ** 3) / 16

        #alles auf form von x_batch für sequenz und batch
        W_p = W_p.expand_as(X_batch)
        A_S = A_S.expand_as(X_batch)
        d2 = d2.expand_as(X_batch)
        d_K_R = d_K_R.expand_as(X_batch)

        #f_v denom = 1/A_S**2 + 3 * (d2**2 * torch.tan(steigungswinkel + gewindereibwinkel)**2) / (4 * W_p**2)
        fv_denom = (1 / (A_S ** 2)+ 3 * (d2 ** 2 * torch.tan(steigungswinkel + gewindereibwinkel) ** 2)/ (4 * W_p ** 2))

        #F_V = zul_spannung / fv_denom.sqrt()
        F_V = self.zul_spannung / torch.sqrt(fv_denom)

        # M = F_V * ((d2 / 2) * torch.tan(steigungswinkel + gewindereibwinkel) + mu_K * d_K_R)
        torque_phys = F_V * ((d2 / 2) * torch.tan(steigungswinkel + gewindereibwinkel)+ mu_K * d_K_R)

        #loss über sequenz dann
        loss = F.mse_loss(torque_phys, X_batch)

        return loss

In [10]:
################################################################
#  1d fourier layer
################################################################
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1):
        super(SpectralConv1d, self).__init__()

        """
        1D Fourier layer. It does FFT, linear transform, and Inverse FFT.    
        """

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1  #Number of Fourier modes to multiply, at most floor(N/2) + 1

        self.scale = (1 / (in_channels*out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, dtype=torch.cfloat))

    # Complex multiplication
    def compl_mul1d(self, input, weights):
        # (batch, in_channel, x ), (in_channel, out_channel, x) -> (batch, out_channel, x)
        return torch.einsum("bix,iox->box", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        #Compute Fourier coeffcients up to factor of e^(- something constant)
        x_ft = torch.fft.rfft(x)

        # Multiply relevant Fourier modes
        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-1)//2 + 1,  device=x.device, dtype=torch.cfloat)
        out_ft[:, :, :self.modes1] = self.compl_mul1d(x_ft[:, :, :self.modes1], self.weights1)

        #Return to physical space
        x = torch.fft.irfft(out_ft, n=x.size(-1))
        return x

class FNO1d(nn.Module):
    def __init__(self, modes, width, n_classes):
        super(FNO1d, self).__init__()

        """
        The overall network. It contains 4 layers of the Fourier layer.
        1. Lift the input to the desire channel dimension by self.fc0 .
        2. 4 layers of the integral operators u' = (W + K)(u).
            W defined by self.w; K defined by self.conv .
        3. Project from the channel space to the output space by self.fc1 and self.fc2 .
        
        input: the solution of the initial condition and location (a(x), x)
        input shape: (batchsize, x=s, c=2)
        output: the solution of a later timestep
        output shape: (batchsize, x=s, c=1)
        """

        self.modes1 = modes
        self.width = width
        self.padding = 2 # pad the domain if input is non-periodic
        self.fc0 = nn.Linear(2, self.width) # input channel is 2: (a(x), x)

        self.conv0 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv1 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv2 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv3 = SpectralConv1d(self.width, self.width, self.modes1)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)

        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        grid = self.get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x = self.fc0(x)
        x = x.permute(0, 2, 1)
        # x = F.pad(x, [0,self.padding]) # pad the domain if input is non-periodic

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = x1 + x2

        # 1. Global Pooling: Average across the 1000 time steps
        # This collapses the sequence dimension
        x = torch.mean(x, dim=-1) # New shape: (batch, width)

        # x = x[..., :-self.padding] # pad the domain if input is non-periodic
        #x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return x

    def get_grid(self, shape, device):
        batchsize, size_x = shape[0], shape[1]
        gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
        gridx = gridx.reshape(1, size_x, 1).repeat([batchsize, 1, 1])
        return gridx.to(device)
    
class FNO2d(nn.Module):
    def __init__(self, modes, width, n_classes, input_channels = 1):
        super(FNO2d, self).__init__()

        """
        The overall network. It contains 4 layers of the Fourier layer.
        1. Lift the input to the desire channel dimension by self.fc0 .
        2. 4 layers of the integral operators u' = (W + K)(u).
            W defined by self.w; K defined by self.conv .
        3. Project from the channel space to the output space by self.fc1 and self.fc2 .
        
        input: the solution of the initial condition and location (a(x), x)
        input shape: (batchsize, x=s, c=2)
        output: the solution of a later timestep
        output shape: (batchsize, x=s, c=1)
        """

        self.modes1 = modes
        self.width = width
        self.padding = 2 # pad the domain if input is non-periodic
        self.fc0 = nn.Linear(input_channels +1, self.width) # input channel is flexibel for more features e.g. torque + angle

        self.conv0 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv1 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv2 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv3 = SpectralConv1d(self.width, self.width, self.modes1)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)

        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        grid = self.get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x = self.fc0(x)
        x = x.permute(0, 2, 1)
        # x = F.pad(x, [0,self.padding]) # pad the domain if input is non-periodic

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = x1 + x2

        # 1. Global Pooling: Average across the 1000 time steps
        # This collapses the sequence dimension
        x = torch.mean(x, dim=-1) # New shape: (batch, width)

        # x = x[..., :-self.padding] # pad the domain if input is non-periodic
        #x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return x

    def get_grid(self, shape, device):
        batchsize, size_x = shape[0], shape[1]
        gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
        gridx = gridx.reshape(1, size_x, 1).repeat([batchsize, 1, 1])
        return gridx.to(device)

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#config 
ntrain = 1000
ntest = 100

sub = 2**3 #subsampling rate
h = 2**13 // sub #total grid size divided by the subsampling rate
s = h

batch_size = 20
learning_rate = 0.001

epochs = 500
step_size = 50
gamma = 0.5

modes = 16
width = 64
model = FNO1d(modes, width, n_classes=8).to(device)

In [12]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

# 3CV fold evaluation

In [13]:
import math
from sklearn.metrics import f1_score
x_data = np.array(df['torque_values'].tolist())[..., np.newaxis]
y_data = np.array(df['class_values'].tolist())
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

angle = np.array(df['angle_values'].tolist())[..., np.newaxis]
phase = np.array(df['step_values'].tolist())[..., np.newaxis]

"""angle = np.transpose(angle, (0, 2, 1))
phase = np.transpose(phase, (0, 2, 1))
x_data = np.transpose(x_data, (0, 2, 1))"""

X_train_full, X_test, angle_train_full, angle_test, phase_train_full, phase_test, y_train_full, y_test = train_test_split(
    x_data, angle, phase, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=81)
cv_scores = []

train_losses = []
train_physics_losses = []
val_losses = []
val_physics_losses = []
val_f1_scores = []

best_val_f1 = -np.inf
best_model_state = None
lambda_phys = 1.0
best_overall_f1 = -np.inf
best_overall_model_state = None

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):

    print(f"Fold {fold+1}")

    X_train_fold = X_train_full[train_idx]
    y_train_fold = y_train_full[train_idx]
    angle_train_fold = angle_train_full[train_idx]
    phase_train_fold = phase_train_full[train_idx]

    X_val_fold = X_train_full[val_idx]
    y_val_fold = y_train_full[val_idx]
    angle_val_fold = angle_train_full[val_idx]
    phase_val_fold = phase_train_full[val_idx]

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train_fold, dtype=torch.float32),
            torch.tensor(angle_train_fold, dtype=torch.float32),
            torch.tensor(phase_train_fold, dtype=torch.float32),
            torch.tensor(y_train_fold, dtype=torch.long)
        ),
        batch_size=32,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_val_fold, dtype=torch.float32),
            torch.tensor(angle_val_fold, dtype=torch.float32),
            torch.tensor(phase_val_fold, dtype=torch.float32),
            torch.tensor(y_val_fold, dtype=torch.long)
        ),
        batch_size=32,
        shuffle=False
    )

    model = FNO1d(modes, width, n_classes=8).to(device)

    criterion = nn.CrossEntropyLoss()
    physics_loss = Anziehdrehmoment().to(device)
    optimizer = torch.optim.AdamW(list(model.parameters()) + list(physics_loss.parameters()), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    earlystop = EarlyStopper(patience=7, min_delta=0.001)

    epochs = 50

    train_losses = []
    train_physics_losses = []
    val_losses = []
    val_physics_losses = []
    val_f1_scores = []
    best_val_f1 = -np.inf
    best_model_state = None


    for epoch in range(epochs):

        model.train()
        total_loss = 0.0
        total_phys = 0.0

        for X_batch, angle_batch, phase_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            angle_batch = angle_batch.to(device)
            phase_batch = phase_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)
            loss_class = criterion(outputs, y_batch)

            loss_physics = physics_loss(outputs, phase_batch, angle_batch, X_batch, y_batch)
            loss = loss_class + lambda_phys * loss_physics
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_phys += loss_physics.item()

        avg_train_loss = total_loss / len(train_loader)
        avg_train_phys_loss = total_phys / len(train_loader)
        train_losses.append(avg_train_loss)
        train_physics_losses.append(avg_train_phys_loss)

        model.eval()

        val_loss = 0.0
        val_physics_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():

            for X_val_batch, angle_val_batch, phase_val_batch, y_val_batch in val_loader:

                X_val_batch = X_val_batch.to(device)
                angle_val_batch = angle_val_batch.to(device)
                phase_val_batch = phase_val_batch.to(device)
                y_val_batch = y_val_batch.to(device)

                outputs = model(X_val_batch)

                loss_class = criterion(outputs, y_val_batch)
                loss_physics = physics_loss(outputs, phase_val_batch, angle_val_batch, X_val_batch, y_val_batch)

                loss = loss_class + lambda_phys * loss_physics

                val_loss += loss.item()
                val_physics_loss += loss_physics.item()

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_val_batch.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        avg_val_phys_loss = val_physics_loss / len(val_loader)
        val_physics_losses.append(avg_val_phys_loss)

        val_f1 = f1_score(all_labels, all_preds, average="macro")
        val_f1_scores.append(val_f1)

        scheduler.step(avg_val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict()

        if earlystop.early_stop(avg_val_loss):
            print("Early stopping triggered")
            break

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Train Physics Loss: {train_physics_losses[-1]:.4f}, Val Physics Loss: {val_physics_losses[-1]:.4f}, Val F1 Score: {val_f1:.4f}")

    cv_scores.append(best_val_f1)

    if best_val_f1 > best_overall_f1:
        best_overall_f1 = best_val_f1
        best_overall_model_state = best_model_state

print(f"CV f1 mean avg score: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")

Fold 1
Epoch 1/50, Train Loss: 1.9484, Val Loss: 1.8131, Train Physics Loss: 0.1051, Val Physics Loss: 0.1038, Val F1 Score: 0.2409
Epoch 2/50, Train Loss: 1.7788, Val Loss: 1.7712, Train Physics Loss: 0.1047, Val Physics Loss: 0.1038, Val F1 Score: 0.2741
Epoch 3/50, Train Loss: 1.7540, Val Loss: 1.7412, Train Physics Loss: 0.1048, Val Physics Loss: 0.1038, Val F1 Score: 0.2666
Epoch 4/50, Train Loss: 1.7392, Val Loss: 1.7439, Train Physics Loss: 0.1047, Val Physics Loss: 0.1039, Val F1 Score: 0.2724
Epoch 5/50, Train Loss: 1.7206, Val Loss: 1.7372, Train Physics Loss: 0.1046, Val Physics Loss: 0.1038, Val F1 Score: 0.2523
Epoch 6/50, Train Loss: 1.6979, Val Loss: 1.7147, Train Physics Loss: 0.1046, Val Physics Loss: 0.1040, Val F1 Score: 0.3038
Epoch 7/50, Train Loss: 1.6694, Val Loss: 1.6903, Train Physics Loss: 0.1047, Val Physics Loss: 0.1038, Val F1 Score: 0.3094
Epoch 8/50, Train Loss: 1.6220, Val Loss: 1.6354, Train Physics Loss: 0.1046, Val Physics Loss: 0.1041, Val F1 Score: 

In [14]:
model = FNO1d(modes, width, n_classes=8).to(device)
final_model = model
final_model.to(device)
final_model.load_state_dict(best_overall_model_state)
final_model.eval()

test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),torch.tensor(y_test, dtype=torch.long)),batch_size=32,shuffle=False)

all_preds = []
all_labels = []
test_loss = 0.0

criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:

        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)
        outputs = final_model(X_test_batch)
        loss = criterion(outputs, y_test_batch)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"Final Test F1 Macro: {test_f1:.4f}")

Final Test F1 Macro: 0.3852


In [ ]:
import torch
print(torch.__version__)
